In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#procesing datasets
%pip install kaggle
%kaggle datasets download apollo2506/eurosat-dataset
%mkdir -p "/content/drive/MyDrive/Datasets"
%cp eurosat-dataset.zip "/content/drive/MyDrive/Datasets/"


In [ ]:
#copy dataset to the content folder
%cp "/content/drive/MyDrive/Datasets/eurosat-dataset.zip" /content/
%unzip -q -n eurosat-dataset.zip

In [ ]:
import random
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import rasterio

class Validation:
    def __init__(self, dir_path, expected_shape=(64, 64)):
        self.dir_path = Path(dir_path)
        self.expected_shape = expected_shape
        self.valid_images = []

    def tree_structure(self, current_path=None, indent=""):
        if current_path is None:
            current_path = self.dir_path

        if current_path.is_dir():
            print(indent + current_path.name)
            for child in current_path.iterdir():
                if child.is_dir():
                    self.tree_structure(child, indent + "  ")

    def data_report(self):
        print("Scanning...\n")

        total_samples = 0
        widths = []
        heights = []
        bad_format = []
        corrupted = []

        # Reset the valid images list in case we run the report twice
        self.valid_images = []

        for image_path in self.dir_path.rglob("*.tif"):
            if image_path.is_file():
                if image_path.parent.parent != self.dir_path:
                    continue

                total_samples += 1

                try:
                    with rasterio.open(image_path) as dataset:
                        w, h = dataset.width, dataset.height

                        widths.append(w)
                        heights.append(h)

                        #if image resolution is not valid then skip,using this allows to pass only valid data in dataloader later
                        if (w, h) != self.expected_shape:
                            bad_format.append(image_path)
                        else:
                            self.valid_images.append(image_path)

                except Exception:
                    corrupted.append(image_path)
                    continue

        print("Data Report:\n")
        print(f"Total samples: {total_samples}")
        print(f"Valid samples: {len(self.valid_images)}")
        print(f"Corrupted files: {len(corrupted)}")
        print(f"Invalid Format (Not {self.expected_shape[0]}x{self.expected_shape[1]}): {len(bad_format)}")

        if bad_format:
            print("First 3 invalid files:", [p.name for p in bad_format[:3]])

        if widths and heights:
            print("\nWidth statistic:")
            print(f"Min: {min(widths)} px | max: {max(widths)} px | mean: {np.mean(widths):.2f} px")
            print("Height statistic:")
            print(f"min: {min(heights)} px | max: {max(heights)} px | mean: {np.mean(heights):.2f} px\n")

        self.random_plot()

    def random_plot(self):
        if self.valid_images:
            random_sample = random.choice(self.valid_images)

            with rasterio.open(random_sample) as dataset:
              #In Sentinel-2 red green and blue channels are 2,3,4
                r = dataset.read(4)
                g = dataset.read(3)
                b = dataset.read(2)
              #normalization colors  for matplotlib
                rgb = np.dstack((r, g, b))
                rgb = rgb / np.max(rgb)

            plt.figure(figsize=(6,6))
            plt.imshow(rgb)
            plt.title(f"Random TIF (RGB View): {random_sample.parent.name}")
            plt.axis('off')
            plt.show()
        else:
            print("No valid images found to plot")

    def generate_nir_mask(self, tif_path, n_std=1.0):
        path = Path(tif_path)
        if not path.is_file():
            print(f"Error: File not found at {path}")
            return None

        try:
            with rasterio.open(path) as dataset:
                #next 4 lines is defining how each image in class is sensetive to NIR to adapt for each image for better model perfomance
                nir_band = dataset.read(8)
                img_mean = np.mean(nir_band)
                img_std = np.std(nir_band)

                adaptive_threshold = img_mean + (n_std * img_std)

                print(f"Image stats: min brightness: {np.min(nir_band)} | max: {np.max(nir_band)} | mean: {np.mean(nir_band):.0f}")
                print(f"Adaptive threshold set to: {adaptive_threshold:.0f} (Using n={n_std})")
                nir_mask = nir_band > adaptive_threshold

            fig, axes = plt.subplots(1, 2, figsize=(10, 5))
            axes[0].imshow(nir_band, cmap='gray')
            axes[0].set_title("Raw NIR Band")
            axes[0].axis('off')

            axes[1].imshow(nir_mask, cmap='binary_r')
            axes[1].set_title(f"Mask mean + ({n_std}*std)")
            axes[1].axis('off')

            plt.tight_layout()
            plt.show()

            return nir_mask

        except Exception as e:
            print(f"Error processing {path}: {e}")
            return None

    def random_nir_mask(self, n_std=1.0):
        if self.valid_images:
            random_sample = random.choice(self.valid_images)
            print(f"Generating mask for: {random_sample.name} (category: {random_sample.parent.name})")
            return self.generate_nir_mask(random_sample, n_std=n_std)
        else:
            print("No valid .tif files found.")
            return None

    def compute_dataset_stats(self, custom_image_list=None):
        print("\nComputing global mean and std for RGB + NIR channels...")

        target_images = custom_image_list if custom_image_list is not None else self.valid_images

        if not target_images:
            print("No valid images found to process!")
            return None, None

        channel_sum = np.zeros(4, dtype=np.float64)
        channel_sum_squared = np.zeros(4, dtype=np.float64)
        total_pixels = 0

        for idx, image_path in enumerate(target_images):
            if idx > 0 and idx % 5000 == 0:
                print(f"Processed {idx} / {len(target_images)} images...")

            with rasterio.open(image_path) as dataset:
                r = dataset.read(4)
                g = dataset.read(3)
                b = dataset.read(2)
                nir = dataset.read(8)

                img = np.stack([r, g, b, nir], axis=0).astype(np.float64)

                pixels_in_image = img.shape[1] * img.shape[2]
                total_pixels += pixels_in_image

                # axis=(1, 2) means we sum across the width and height, leaving an array of 4 totals
                channel_sum += img.sum(axis=(1, 2))

                channel_sum_squared += (img ** 2).sum(axis=(1, 2))

        global_mean = channel_sum / total_pixels

        global_variance = (channel_sum_squared / total_pixels) - (global_mean ** 2)

        global_std = np.sqrt(global_variance)

        print("\n Global dataset statistics:")
        print(f"Bands calculated: red, green, blue, NIR")
        print(f"Mean: {np.round(global_mean, 2).tolist()}")
        print(f"std:  {np.round(global_std, 2).tolist()}")

        return global_mean, global_std



In [ ]:
#Cause we are using a NIR band to,we cannot get a 4 layer tensor without data manipulation first
import torch
from torch.utils.data import Dataset
import torchvision.transforms as T
import rasterio
import numpy as np

class EuroSATDataset(Dataset):
    def __init__(self, valid_image_paths, dataset_mean, dataset_std):
        self.image_path = valid_image_paths
        self.normalize = T.Normalize(mean=dataset_mean, std=dataset_std)

        unique_classes = sorted(list(set([p.parent.name for p in self.image_path])))

        self.class_to_idx = {class_name: idx  for idx, class_name in enumerate(unique_classes)}
        print(f"Found {len(unique_classes)} classes: {self.class_to_idx}")

    def __len__(self):
        return len(self.image_path)

    def __getitem__(self, idx):
        img_path = self.image_path[idx]

        with rasterio.open(img_path) as dataset:
            r = dataset.read(4)
            g = dataset.read(3)
            b = dataset.read(2)
            nir = dataset.read(8)

        img_array = np.stack([r, g, b, nir], axis=0)
        img_tensor = torch.from_numpy(img_array).float()
        img_tensor = self.normalize(img_tensor)

        label_str = img_path.parent.name
        label_idx = self.class_to_idx[label_str]

        return img_tensor, label_idx

In [ ]:
validator = Validation("/content/EuroSATallBands", expected_shape=(64, 64))


In [ ]:
validator.data_report()

In [ ]:
#spliting for correct mean and std computation,to use in data loader
from sklearn.model_selection import train_test_split
all_clean_files = validator.valid_images


train_val_files, test_files = train_test_split(all_clean_files, test_size=0.15, random_state=42)

#double split for validation chunck to be involved in dataset
train_files, val_files = train_test_split(train_val_files, test_size=(0.15 / 0.85), random_state=42)

print(f"\nSplit complete:")
print(f"Train: {len(train_files)} | val: {len(val_files)} | test: {len(test_files)}")

mean, std = validator.compute_dataset_stats(custom_image_list=train_files)

In [ ]:
#Make datasets and  Dataloaders
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader


train_dataset = EuroSATDataset(train_files, dataset_mean=mean, dataset_std=std)
val_dataset = EuroSATDataset(val_files, dataset_mean=mean, dataset_std=std)
test_dataset = EuroSATDataset(test_files, dataset_mean=mean, dataset_std=std)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

print("\nData loaders are ready to use")

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class EuroSATResNet(nn.Module):
    def __init__(self):
        super().__init__()

        self.engine = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

        for block in [self.engine.layer1, self.engine.layer2]:
            for param in block.parameters():
                param.requires_grad = False


        old_conv = self.engine.conv1

        new_conv = nn.Conv2d(
            in_channels=4,
            out_channels=old_conv.out_channels,
            kernel_size=old_conv.kernel_size,
            stride=old_conv.stride,
            padding=old_conv.padding,
            bias=False
        )

        #Copy original RGB weights
        new_conv.weight.data[:, :3, :, :] = old_conv.weight.data

        # Average the RGB weights to create the NIR channel head-start
        nir_weights = old_conv.weight.data.mean(dim=1, keepdim=True)
        new_conv.weight.data[:, 3:4, :, :] = nir_weights

        self.engine.conv1 = new_conv


        self.engine.fc = nn.Linear(in_features=512, out_features=10)

    def forward(self, x):
        return self.engine(x)


In [ ]:
model = EuroSATResNet()
print("Creating a model was succesfull")

In [ ]:
#defining computational hardware to make training more effective
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
#Next step is to define the model metrics(loss function) & optimazer to use them in training
base_lr = 0.001

optimizer = torch.optim.Adam([
    #define a Two-stage fine tuning of layers for perfomance
    #the most well pretrained layers,devide by 100
    {"params": model.engine.layer1.parameters(), "lr": base_lr / 100},
    {"params": model.engine.layer2.parameters(), "lr": base_lr / 100},

    #pre_trained weight,  devide by 10 cause it pretrained less than layer1,2
    {"params": model.engine.layer3.parameters(), "lr": base_lr / 10},
    {"params": model.engine.layer4.parameters(), "lr": base_lr / 10},

    #brande new weight,leave learning rate by default
    {"params": model.engine.conv1.parameters(), "lr": base_lr},
    {"params": model.engine.fc.parameters(), "lr": base_lr},


])
criterion = nn.CrossEntropyLoss()

In [ ]:
#Time  for training loop!

from tqdm import tqdm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Training on device: {device}")

epochs = 20

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1} / {epochs}")

    model.train() # Tells layers like Dropout/BatchNorm to act in training mode
    train_loss = 0.0
    train_correct = 0

    #Loop through the training batches
    for images, labels in tqdm(train_loader, desc="Training"):
        #Move data to the GPU
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # Track metrics
        train_loss += loss.item() * images.size(0)
        _, predictions = torch.max(outputs, 1) # Get the index of the highest probability
        train_correct += torch.sum(predictions == labels.data)

    avg_train_loss = train_loss / len(train_loader.dataset)
    train_accuracy = train_correct.double() / len(train_loader.dataset)



    model.eval() #Tells the model we are just testing, NO learning allowed
    val_loss = 0.0
    val_correct = 0

    #Disable gradient tracking to save massive amounts of memory and time
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc="Validating"):
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            _, predictions = torch.max(outputs, 1)
            val_correct += torch.sum(predictions == labels.data)

    avg_val_loss = val_loss / len(val_loader.dataset)
    val_accuracy = val_correct.double() / len(val_loader.dataset)

    print(f"Train Loss:{avg_train_loss:.4f} | Train Acc:{train_accuracy:.4f}")
    print(f"Val Loss:{avg_val_loss:.4f} | Val Acc:{val_accuracy:.4f}")

print("\nTraining Complete!Now you can evaluate on the test_loader.")


In [ ]:
#The test set comes in to take a true metrics
model.eval()
test_loss = 0.0
test_correct = 0
total_samples = len(test_loader.dataset)

print("\nStarting final test on training loop...")

with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        test_loss += loss.item() * images.size(0)

        _, predictions = torch.max(outputs, 1)

        test_correct += torch.sum(predictions == labels.data)

avg_test_loss = test_loss / total_samples
test_accuracy = test_correct.double() / total_samples

print("Final results:")
print(f"Test loss:{avg_test_loss:.4f}")
print(f"Test accuracy:{test_accuracy * 100:.2f}%")


In [ ]:
%pip install grad-cam

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import random
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

all_preds = []
all_true = []

model.eval()
with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predictions = torch.max(outputs, 1)

        all_preds.extend(predictions.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

class Evaluation:
    def __init__(self, y_true, y_pred, class_names):
        self.y_true = y_true
        self.y_pred = y_pred
        self.class_names = class_names

    def plot_confusion_matrix(self):
        cm = confusion_matrix(self.y_true, self.y_pred, normalize='true')

        fig, ax = plt.subplots(figsize=(12, 12))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=self.class_names)
        disp.plot(cmap='Blues', ax=ax, xticks_rotation='vertical', values_format='.2f')

        plt.title("EuroSAT 4-channel ResNet confusion matrix", fontsize=16, pad=20)
        plt.tight_layout()
        plt.show()

    def print_confidence_interval(self, confidence_level=0.95):
        n = len(self.y_true)
        correct_predictions = sum(1 for true, pred in zip(self.y_true, self.y_pred) if true == pred)
        p = correct_predictions / n

        z =  1.96

        margin_of_error = z * np.sqrt(p * (1-p) / n)

        lower_bound = (p - margin_of_error) * 100
        upper_bound = (p + margin_of_error) * 100

        accuracy_pct = p * 100

        print(f"Test Set Size (n): {n} images")
        print(f"Estimate accuracy: {accuracy_pct:.2f}%")
        print(f"95% confidence interval: [{lower_bound:.2f}%, {upper_bound:.2f}%]")
        print(f"Margin of error is: ±{margin_of_error * 100:.2f}%")


    def class_report(self):
        cr = classification_report(self.y_true, self.y_pred, target_names=self.class_names, digits=2)
        print(cr)

    def attention_heatmap(self, model, input_tensor, true_label):
        with torch.no_grad():
            output = model(input_tensor)
            pred_idx = torch.argmax(output, dim=1).item()
            pred_name = self.class_names[pred_idx]
            true_name = self.class_names[true_label]

        target_layers = [model.engine.layer4[-1]]
        cam = GradCAM(model=model, target_layers=target_layers)

        targets = [ClassifierOutputTarget(true_label)]
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

        rgb_image = input_tensor[0, :3, :, :].cpu().numpy().transpose(1, 2, 0)
        p2, p98 = np.percentile(rgb_image, (2, 98))
        rgb_image = np.clip(rgb_image, p2, p98)
        rgb_image = (rgb_image - rgb_image.min()) / (rgb_image.max() - rgb_image.min())

        visualization = show_cam_on_image(rgb_image, grayscale_cam, use_rgb=True)

        fig, ax = plt.subplots(1, 2, figsize=(8, 6))
        ax[0].imshow(visualization)
        ax[0].set_title(f"Grad-CAM Attention: {self.class_names[true_label]}", fontsize=14)
        ax[0].axis('off')

        ax[1].imshow(rgb_image)
        ax[1].set_title(f"Original Photo: {self.class_names[true_label]}", fontsize=14)
        ax[1].axis('off')
        plt.tight_layout()
        plt.show()

    def full_eval_report(self, model, input_tensor, true_label):
        self.plot_confusion_matrix()
        print("\n classification report:")
        self.class_report()
        print("\nintervals of confidence:")
        self.print_confidence_interval()
        print("\nheatmap:")
        self.attention_heatmap(model, input_tensor, true_label)




In [ ]:
eurosat_classes = [
    'AnnualCrop', 'Forest', 'HerbaceousVeg', 'Highway',
    'Industrial', 'Pasture', 'PermanentCrop', 'Residential',
    'River', 'SeaLake'
]

evaluator = Evaluation(all_true, all_preds, eurosat_classes)

sample_images, sample_labels = next(iter(test_loader))
random_idx = random.randint(0, len(sample_images)-1)
sample_input = sample_images[random_idx].unsqueeze(0).to(device)
sample_label = sample_labels[random_idx].item()

evaluator.full_eval_report(model, sample_input, sample_label)


In [ ]:
from pathlib  import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/EuroSAT_Project")

MODELS_DIR = PROJECT_ROOT / "models"
DATA_DIR = PROJECT_ROOT / 'data'

MODELS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

save_path = MODELS_DIR / 'resnet18_4channel_v1.pth'

#torch.save(model.state_dict(), save_path)
print(f"Models succesfully saved  to {save_path}")

In [ ]:
import torch
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = EuroSATResNet()

PROJECT_ROOT = Path("/content/drive/MyDrive/EuroSAT_Project")
load_path = PROJECT_ROOT / 'models' / 'resnet18_4channel_v1.pth'

model.load_state_dict(torch.load(load_path, map_location=device))

model = model.to(device)
model.eval()

print("Pre-trained model successfully loaded")